## Set up

**libraries**

In [ ]:
import pandas as pd

**paths**

In [ ]:
PATH = "../../data/deep learning project data/"
INVS = "/invoices/invoices-1.csv"
ORDS = "/predocs/orders-1.csv"
EMFS = "/predocs/EMFs-1.csv"
OUT  = "documents coin 1.csv"

**constants**

In [ ]:
VAL    = "volume"
FYEAR  = "fund year"
YEAR   = "year"
MONTH  = "month"
DOC    = "doc"
GROUP  = ["doc", "tressure group"]

In [ ]:
invoices = pd.read_csv(PATH + INVS,  usecols = [VAL, YEAR, MONTH,  * GROUP])
orders   = pd.read_csv(PATH + ORDS,  usecols = [VAL, YEAR, MONTH,  * GROUP, FYEAR])
emfs     = pd.read_csv(PATH + EMFS,  usecols = [VAL, YEAR, MONTH,  * GROUP, FYEAR])

<br>

## Preprocessing

filtering for spesific documnets

In [ ]:
emfs = emfs[emfs[DOC].isin([50, 51, 60, 61, 78, 79])]

summing series that sould go together

In [ ]:
emfs.loc[emfs[DOC].isin([50, 51]), DOC] = 50
emfs.loc[emfs[DOC].isin([60, 61]), DOC] = 60
emfs[DOC]
invoices.loc[invoices[DOC].isin(["ZY", "ZF"]), DOC] = "RE"
invoices.loc[invoices[DOC].isin(["KG"]), DOC] = "KR"

main preprocessing steps

In [ ]:
# Get the effective date from which the value can flow

def update_dates(table: pd.DataFrame) -> pd.DataFrame:

    mask = table[YEAR] < table[FYEAR]
    table.loc[mask, YEAR] = table.loc[mask, FYEAR]
    table.loc[mask, MONTH] = 1
    table = table.drop(columns = [FYEAR])
    
    return table


# Sum duplicates, For each date and group there is no need for more than one record

def unique_key(table: pd.DataFrame) -> pd.DataFrame:

    return table.groupby(table.columns.difference([VAL]).tolist(), as_index = False)[VAL].sum()


# Cumulative sum for each group over time

def accumulate(table: pd.DataFrame) -> pd.DataFrame:

    table = table.sort_values(by = [* GROUP, YEAR, MONTH])
    table[VAL] = table.groupby([ * GROUP])[VAL].cumsum()

    return table


# Clip the years to match the invoices date range

def clip(table: pd.DataFrame) -> pd.DataFrame:

    table = table[table[YEAR].between(invoices[YEAR].min(), invoices[YEAR].max())]
    table = table[(table[YEAR] < invoices[YEAR].max()) | (table[MONTH] <= invoices[MONTH].max())]

    return table


def pipeline(table: pd.DataFrame) -> pd.DataFrame:

    table = update_dates(table)
    table = unique_key(table)
    table = accumulate(table)
    table = clip(table)

    return table

In [ ]:
emfs = pipeline(emfs)
orders = pipeline(orders)

visualizing the data to see that everyting makes sense

In [ ]:
def plot_series(table: pd.DataFrame, name: str):

    table.groupby([YEAR, MONTH])[VAL].sum().plot(title = name)

In [ ]:
plot_series(emfs, "EMFs")
plot_series(orders, "Orders")
plot_series(invoices, "Invoices")

<br>

## Saving all data

In [ ]:
combined = pd.concat([invoices, orders, emfs], ignore_index = True)

In [ ]:
combined.head()

In [ ]:
combined.tail()

In [ ]:
combined.to_csv(PATH + OUT, index = False)